# Árvore de Decisão — Titanic

Este notebook repete as etapas do notebook **AD-Restaurante** para a base do Titanic: leitura, limpeza, visualização, codificação, treino, avaliação e extração das regras.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Leitura da base

In [ ]:
# Funciona na pasta Lista02 e também no Google Colab.
candidatos = [
    Path('titanic completo.csv'),
    Path('/content/sample_data/titanic completo.csv'),
    Path('/content/titanic completo.csv'),
]
arquivo_titanic = next((arquivo for arquivo in candidatos if arquivo.exists()), None)
if arquivo_titanic is None:
    raise FileNotFoundError('Coloque o arquivo titanic completo.csv na pasta atual ou em /content/sample_data.')

base_original = pd.read_csv(arquivo_titanic)
print('Dimensões da base original:', base_original.shape)
base_original.head()

## 2. Tratamento dos dados ausentes

In [ ]:
print('Valores ausentes na base original:')
display(base_original.isna().sum().to_frame('ausentes'))

# Como ainda não foi estudado tratamento de dados ausentes, removemos
# os atributos com muitas ausências.
colunas_removidas = ['boat', 'body', 'cabin', 'home.dest']
base = base_original.drop(columns=colunas_removidas).copy()

print('Colunas restantes:')
print(base.columns.tolist())
print('\nValores ausentes antes da remoção das linhas:')
display(base.isna().sum().to_frame('ausentes'))

# Remove as instâncias que ainda possuem algum valor ausente.
base = base.dropna().reset_index(drop=True)
print('Dimensões depois da limpeza:', base.shape)
print('Valores ausentes restantes:', int(base.isna().sum().sum()))

## 3. Visualização e distribuição dos atributos

In [ ]:
print('Distribuição da classe (0 = morreu; 1 = sobreviveu):')
display(base['survived'].value_counts().sort_index().to_frame('quantidade'))

print('Distribuição por sexo:')
display(pd.crosstab(base['sex'], base['survived'], margins=True))

print('Distribuição por classe do passageiro:')
display(pd.crosstab(base['pclass'], base['survived'], margins=True))

print('Distribuição por porto de embarque:')
display(pd.crosstab(base['embarked'], base['survived'], margins=True))

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(13, 9))
sns.countplot(data=base, x='survived', ax=eixos[0, 0])
eixos[0, 0].set_title('Classe: 0 = morreu; 1 = sobreviveu')
sns.countplot(data=base, x='sex', hue='survived', ax=eixos[0, 1])
eixos[0, 1].set_title('Sobrevivência por sexo')
sns.countplot(data=base, x='pclass', hue='survived', ax=eixos[1, 0])
eixos[1, 0].set_title('Sobrevivência por classe')
sns.countplot(data=base, x='embarked', hue='survived', ax=eixos[1, 1])
eixos[1, 1].set_title('Sobrevivência por embarque')
plt.tight_layout()
plt.show()

## 4. Separação e codificação dos atributos

In [ ]:
# name e ticket continuam na base limpa, mas não são usados no modelo,
# pois são identificadores textuais com muitos valores diferentes.
atributos_utilizados = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked'
]
X = base[atributos_utilizados].copy()
y = base['survived'].astype(int).copy()

# sex é binário.
X['sex'] = X['sex'].map({'male': 0, 'female': 1})

# embarked é nominal e recebe One-Hot Encoding, como o atributo Tipo
# no notebook AD-Restaurante.
X = pd.get_dummies(X, columns=['embarked'], dtype=int)

print('Colunas usadas pela árvore:')
print(X.columns.tolist())
X.head()

## 5. Divisão entre treino e teste

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Treino:', X_treino.shape)
print('Teste:', X_teste.shape)

## 6. Treinamento da árvore

In [ ]:
# max_depth=3 mantém a árvore e suas regras interpretáveis.
arvore = DecisionTreeClassifier(
    criterion='entropy', max_depth=3, random_state=42
)
arvore.fit(X_treino, y_treino)
previsoes = arvore.predict(X_teste)
print(f'Acurácia no teste: {accuracy_score(y_teste, previsoes):.2%}')

## 7. Matriz de confusão e métricas

In [ ]:
matriz = confusion_matrix(y_teste, previsoes, labels=[0, 1])
matriz_df = pd.DataFrame(
    matriz,
    index=['Real: Morreu', 'Real: Sobreviveu'],
    columns=['Previsto: Morreu', 'Previsto: Sobreviveu']
)
display(matriz_df)
sns.heatmap(matriz_df, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusão — conjunto de teste')
plt.show()

print(classification_report(
    y_teste, previsoes, labels=[0, 1],
    target_names=['Morreu', 'Sobreviveu'], zero_division=0
))

## 8. Regras obtidas e qualidade de cada regra

In [ ]:
print(export_text(arvore, feature_names=list(X.columns), decimals=2))

In [ ]:
def qualidade_das_regras(modelo, dados, classe_real):
    tabela = pd.DataFrame({
        'folha': modelo.apply(dados),
        'real': classe_real.to_numpy(),
        'previsto': modelo.predict(dados),
    })
    linhas = []
    for numero, (folha, grupo) in enumerate(tabela.groupby('folha'), start=1):
        cobertura = len(grupo)
        acertos = int((grupo['real'] == grupo['previsto']).sum())
        linhas.append({
            'regra': numero,
            'folha': folha,
            'classe': 'Sobreviveu' if int(grupo['previsto'].iloc[0]) == 1 else 'Morreu',
            'cobertura': cobertura,
            'acertos': acertos,
            'qualidade': acertos / cobertura,
        })
    return pd.DataFrame(linhas)

print('Qualidade no treino:')
display(qualidade_das_regras(arvore, X_treino, y_treino).style.format({'qualidade': '{:.2%}'}))
print('Qualidade no teste:')
display(qualidade_das_regras(arvore, X_teste, y_teste).style.format({'qualidade': '{:.2%}'}))

## 9. Visualização da árvore

In [ ]:
plt.figure(figsize=(28, 14))
plot_tree(
    arvore, feature_names=X.columns,
    class_names=['Morreu', 'Sobreviveu'],
    filled=True, rounded=True, fontsize=8
)
plt.title('Árvore de Decisão — Titanic')
plt.tight_layout()
plt.show()